In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load cleaned dataset
df = pd.read_csv('data/diabetic_data_cleaned.csv')

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

Dataset loaded successfully!
Shape: (71515, 44)


In [2]:
# Combine prior hospital visits into one powerful feature
# This captures how 'frequent' a patient is in the healthcare system

df['hospital_utilization_score'] = (df['number_inpatient'] + 
                                     df['number_emergency'] + 
                                     df['number_outpatient'])

print("Hospital Utilization Score stats:")
print(df['hospital_utilization_score'].describe())
print(f"\nCorrelation with readmission: {df['hospital_utilization_score'].corr(df['readmitted']):.4f}")

Hospital Utilization Score stats:
count    71515.000000
mean         0.561463
std          1.431308
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max         49.000000
Name: hospital_utilization_score, dtype: float64

Correlation with readmission: 0.0584


In [3]:
# From EDA we saw insulin changes affect readmission
# Let's count how many medications were changed during the visit
# Medication columns have values: 'Up', 'Down', 'Steady', 'No'

medication_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
                   'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
                   'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
                   'miglitol', 'troglitazone', 'tolazamide', 'examide',
                   'citoglipton', 'insulin', 'glyburide-metformin',
                   'glipizide-metformin', 'glimepiride-pioglitazone',
                   'metformin-rosiglitazone', 'metformin-pioglitazone']

# Count number of medications changed (Up or Down)
df['num_med_changed'] = (df[medication_cols]
                         .isin(['Up', 'Down'])
                         .sum(axis=1))

# Count number of medications prescribed (not 'No')
df['num_med_prescribed'] = (df[medication_cols]
                            .isin(['Up', 'Down', 'Steady'])
                            .sum(axis=1))

print("Medication Changed stats:")
print(df['num_med_changed'].describe())
print(f"\nCorrelation with readmission: {df['num_med_changed'].corr(df['readmitted']):.4f}")

Medication Changed stats:
count    71515.000000
mean         0.262015
std          0.475574
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          4.000000
Name: num_med_changed, dtype: float64

Correlation with readmission: 0.0224


In [4]:
# Combine clinical complexity into one score
# Higher score = more complex patient = higher risk

df['treatment_intensity'] = (df['time_in_hospital'] + 
                              df['num_lab_procedures'] + 
                              df['num_procedures'] + 
                              df['num_medications'])

print("Treatment Intensity Score stats:")
print(df['treatment_intensity'].describe())
print(f"\nCorrelation with readmission: {df['treatment_intensity'].corr(df['readmitted']):.4f}")

Treatment Intensity Score stats:
count    71515.000000
mean        64.500552
std         25.396530
min          3.000000
25%         48.000000
50%         64.000000
75%         80.000000
max        183.000000
Name: treatment_intensity, dtype: float64

Correlation with readmission: 0.0400


In [5]:
# Patients with many diagnoses + long stays are more complex
df['patient_complexity'] = (df['number_diagnoses'] * df['time_in_hospital'])

print("Patient Complexity Score stats:")
print(df['patient_complexity'].describe())
print(f"\nCorrelation with readmission: {df['patient_complexity'].corr(df['readmitted']):.4f}")

Patient Complexity Score stats:
count    71515.000000
mean        32.450619
std         25.892967
min          1.000000
25%         14.000000
50%         27.000000
75%         45.000000
max        208.000000
Name: patient_complexity, dtype: float64

Correlation with readmission: 0.0568


In [6]:
# diag_1 contains ICD-9 diagnosis codes
# Diabetes codes are in range 250-250.99
# Let's flag if primary diagnosis is diabetes related

def is_diabetes_diagnosis(code):
    try:
        code = str(code).strip()
        if code.startswith('250'):
            return 1
    except:
        pass
    return 0

df['primary_diag_diabetes'] = df['diag_1'].apply(is_diabetes_diagnosis)

print("Primary Diabetes Diagnosis distribution:")
print(df['primary_diag_diabetes'].value_counts())
print(f"\nReadmission rate when primary diag is diabetes: "
      f"{df[df['primary_diag_diabetes']==1]['readmitted'].mean()*100:.2f}%")
print(f"Readmission rate when primary diag is NOT diabetes: "
      f"{df[df['primary_diag_diabetes']==0]['readmitted'].mean()*100:.2f}%")

Primary Diabetes Diagnosis distribution:
primary_diag_diabetes
0    65710
1     5805
Name: count, dtype: int64

Readmission rate when primary diag is diabetes: 9.04%
Readmission rate when primary diag is NOT diabetes: 8.78%


In [7]:
# From EDA we saw most patients are elderly
# Let's create meaningful clinical age groups

def age_risk_group(age):
    if age <= 30:
        return 'Young'
    elif age <= 60:
        return 'Middle'
    elif age <= 75:
        return 'Senior'
    else:
        return 'Elderly'

df['age_risk_group'] = df['age'].apply(age_risk_group)

print("Age Risk Group distribution:")
print(df['age_risk_group'].value_counts())
print("\nReadmission rate by Age Risk Group:")
print(df.groupby('age_risk_group')['readmitted'].mean().sort_values(ascending=False) * 100)

Age Risk Group distribution:
age_risk_group
Senior     34167
Middle     22043
Elderly    13489
Young       1816
Name: count, dtype: int64

Readmission rate by Age Risk Group:
age_risk_group
Elderly    10.149010
Senior      9.476981
Middle      7.140589
Young       6.167401
Name: readmitted, dtype: float64


In [8]:
# Label encode all categorical columns
# First identify them
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print("Categorical columns to encode:")
print(categorical_cols)
print(f"\nTotal: {len(categorical_cols)} columns")

Categorical columns to encode:
['race', 'gender', 'admission_type', 'discharge_disposition', 'admission_source', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'age_risk_group']

Total: 35 columns


In [9]:
le = LabelEncoder()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print("Label encoding complete!")
print(f"\nShape after encoding: {df.shape}")
print(f"\nData types now:")
print(df.dtypes.value_counts())

Label encoding complete!

Shape after encoding: (71515, 51)

Data types now:
int64    51
Name: count, dtype: int64


In [10]:
# Drop individual medication columns since we created
# num_med_changed and num_med_prescribed to capture the same info
# Also drop diag_2 and diag_3 as they are secondary diagnoses

cols_to_drop = ['diag_2', 'diag_3', 'examide', 'citoglipton', 
                'metformin-rosiglitazone', 'metformin-pioglitazone',
                'glimepiride-pioglitazone', 'glipizide-metformin',
                'glyburide-metformin', 'troglitazone', 'tolazamide',
                'tolbutamide', 'acetohexamide', 'chlorpropamide']

# Verify these columns exist before dropping
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

df.drop(columns=cols_to_drop, inplace=True)

print(f"Dropped {len(cols_to_drop)} low value columns!")
print(f"Final shape: {df.shape}")
print(f"\nRemaining columns:")
print(df.columns.tolist())

Dropped 14 low value columns!
Final shape: (71515, 37)

Remaining columns:
['race', 'gender', 'age', 'admission_type', 'discharge_disposition', 'admission_source', 'time_in_hospital', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'number_diagnoses', 'metformin', 'repaglinide', 'nateglinide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'insulin', 'change', 'diabetesMed', 'readmitted', 'hospital_utilization_score', 'num_med_changed', 'num_med_prescribed', 'treatment_intensity', 'patient_complexity', 'primary_diag_diabetes', 'age_risk_group']


In [11]:
# Save final dataset for ML modeling
df.to_csv('data/diabetic_data_features.csv', index=False)

print("Feature engineered dataset saved successfully!")
print(f"\nFinal Shape: {df.shape}")
print(f"\nNew features created:")
print("  ✅ hospital_utilization_score")
print("  ✅ num_med_changed")
print("  ✅ num_med_prescribed")
print("  ✅ treatment_intensity")
print("  ✅ patient_complexity")
print("  ✅ primary_diag_diabetes")
print("  ✅ age_risk_group")
print(f"\nTarget distribution:")
print(df['readmitted'].value_counts())

Feature engineered dataset saved successfully!

Final Shape: (71515, 37)

New features created:
  ✅ hospital_utilization_score
  ✅ num_med_changed
  ✅ num_med_prescribed
  ✅ treatment_intensity
  ✅ patient_complexity
  ✅ primary_diag_diabetes
  ✅ age_risk_group

Target distribution:
readmitted
0    65222
1     6293
Name: count, dtype: int64
